[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/072.%20The%20Capacitated%20VRP%20%28CVRP%29/P72-Tier-8_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 72. The Capacitated VRP (CVRP)

## Tier 8: A Value-Aligned Ethical Framework

### Purpose and scope

This notebook evaluates a supplied capacitated vehicle routing plan using stakeholder utilities, ethical metrics, and hard and soft constraints. It then applies a small set of explicit post-processing heuristics: a capacity-aware workload transfer, followed by 2-opt route improvement. The framework scores and reports the resulting plan; it is not a mathematical-programming solver and does not search over all CVRP plans.

**Inputs:** customer locations and demands, vehicle capacities, an initial set of routes and vehicle assignments, and stakeholder and constraint settings. **Outputs:** a modified plan, its calculated distance and ethical indicators, constraint violations, stakeholder utilities, and a report and visualization.

### Model assumptions

- Customer and depot coordinates are expressed in kilometres in a two-dimensional Euclidean plane. The depot is at \((0,0)\), and each route starts and ends there.
- Demand and vehicle capacity use the same abstract load unit. Every customer is served once, and each vehicle's assigned customers must match its route and fit within its capacity.
- The sample does not provide measured delivery times, wages, prices, noise, or service-area data. Utilities for unavailable metrics contribute zero; this is missing information, not evidence of stakeholder satisfaction.
- Service equity and profit margin are illustrative scenario inputs. Driver hours use an assumed average speed and per-customer service time. The community-impact measure is a simplified distance-based index, not a measured percentage change in congestion.
- The Euclidean distance model omits road networks, traffic, time windows, vehicle-specific emissions, and uncertainty. Accordingly, results are illustrative model outputs, not operational recommendations or empirical impact claims.

### Stakeholders and ethical principles

The example represents customers, drivers, the environment, the community, and the company. Their interests can conflict: for example, shorter routes may improve operating efficiency and reduce modeled emissions, while fair workload distribution may require changing vehicle assignments. Stakeholder priorities and utility weights make some value judgments explicit; they do not make those judgments objectively correct.

The framework tracks fairness, sustainability, transparency, accountability, equity, and responsibility through the defined metrics and reporting. It does not measure every principle directly.

### Solution sequence

1. Calculate metrics for the supplied plan.
2. Attempt one workload-balancing transfer when the load imbalance exceeds the specified threshold and the receiving vehicle has capacity.
3. Improve each route using bounded 2-opt local search.
4. Calculate stakeholder utilities, ethical indicators, and constraint violations for the resulting plan.
5. Report feasibility, scores, and the assumptions that limit their interpretation.

### Mathematical definitions

Let \(R_v\) be the ordered customer sequence assigned to vehicle \(v\), \(q_i\) the demand of customer \(i\), and \(Q_v\) the capacity of vehicle \(v\). Let \(d_{ij}\) be Euclidean distance between locations \(i\) and \(j\), with depot \(0\). The distance of a route is

$$\ell(R_v)=d_{0,R_{v,1}}+\sum_{k=1}^{|R_v|-1}d_{R_{v,k},R_{v,k+1}}+d_{R_{v,|R_v|},0},$$

and the total distance is \(D=\sum_v \ell(R_v)\). The implementation is `EthicalOptimizationEngine._calculate_route_distance`; its coordinate calculation uses `numpy.sqrt`.

For a metric vector \(m\), stakeholder \(s\) has utility

$$U_s(m)=\sum_{j\in M_s}w_{sj}m_j,$$

where \(M_s\) contains the metric names in that stakeholder's `utility_weights` and \(w_{sj}\) is the corresponding weight. Missing metrics contribute zero in `Stakeholder.calculate_utility`. Priority-weighted utilities multiply \(U_s\) by the stakeholder's `priority_level` when calculating the aggregate ethical score.

For a constraint with threshold \(b\), penalty weight \(p\), and observed metric \(x\), the weighted violation is

$$v(x)=p\max(0,b-x)\quad\text{for a metric where higher is better},$$
$$v(x)=p\max(0,x-b)\quad\text{for a metric where lower is better}.$$

Hard constraints make the plan infeasible when their weighted violation is positive. Soft violations reduce the compliance score. The implementation is `EthicalConstraint.evaluate_violation`; the direction of each metric is defined there.

The route-improvement step considers 2-opt moves that reverse a contiguous segment of a route. It accepts shorter routes and repeats the search up to a length-dependent iteration limit. This is a local search, not a guarantee of a globally shortest route. If a route has \(n\) customers, one full pass considers \(O(n^2)\) segment reversals; the notebook bounds the number of passes. The workload heuristic attempts at most one transfer, so it does not solve a general load-balancing problem.

### Interpreting results

A feasibility result means only that the evaluated plan has no recorded hard-constraint violation under these assumptions. A utility or ethical score is a score under the chosen metrics, weights, and normalization—not a universal measure of ethical quality. The code reports calculated plan distance separately from the sample's supplied reference distance because those values are based on different information.

In [ ]:
from dataclasses import dataclass
from enum import Enum
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np

np.random.seed(42)

print("Libraries imported for the ethical CVRP framework.")

In [ ]:
class StakeholderType(Enum):
    CUSTOMERS = "customers"
    DRIVERS = "drivers"
    ENVIRONMENT = "environment"
    COMMUNITY = "community"
    COMPANY = "company"
    SUPPLIERS = "suppliers"


class EthicalPrinciple(Enum):
    FAIRNESS = "fairness"
    SUSTAINABILITY = "sustainability"
    TRANSPARENCY = "transparency"
    ACCOUNTABILITY = "accountability"
    EQUITY = "equity"
    RESPONSIBILITY = "responsibility"


class ConstraintType(Enum):
    WORKLOAD_BALANCE = "workload_balance"
    ENVIRONMENTAL_IMPACT = "environmental_impact"
    SERVICE_EQUITY = "service_equity"
    DRIVER_WELLBEING = "driver_wellbeing"
    COMMUNITY_IMPACT = "community_impact"
    ECONOMIC_VIABILITY = "economic_viability"


@dataclass
class Stakeholder:
    """A stakeholder, its interests, and its weighted utility function."""

    stakeholder_id: int
    stakeholder_type: StakeholderType
    name: str
    interests: List[str]
    utility_weights: Dict[str, float]
    constraints: List[str]
    priority_level: float

    def calculate_utility(self, solution_metrics: Dict[str, float]) -> float:
        """Sum weighted metrics that are available for this stakeholder."""
        return sum(
            weight * solution_metrics.get(metric, 0.0)
            for metric, weight in self.utility_weights.items()
        )


@dataclass
class EthicalConstraint:
    """A threshold constraint with a weighted violation measure."""

    constraint_id: str
    constraint_type: ConstraintType
    description: str
    threshold: float
    penalty_weight: float
    is_hard_constraint: bool = False

    def evaluate_violation(self, solution_metrics: Dict[str, float]) -> float:
        """Return zero when satisfied; otherwise return weighted violation."""
        metric_name = self.constraint_type.value
        if self.constraint_type == ConstraintType.DRIVER_WELLBEING:
            metric_name = "driver_hours"

        if metric_name not in solution_metrics:
            raise KeyError(
                f"Required metric '{metric_name}' is missing for "
                f"constraint '{self.constraint_id}'."
            )

        actual_value = solution_metrics[metric_name]
        higher_is_better = self.constraint_type in {
            ConstraintType.WORKLOAD_BALANCE,
            ConstraintType.SERVICE_EQUITY,
            ConstraintType.ECONOMIC_VIABILITY,
        }

        if higher_is_better:
            raw_violation = max(0.0, self.threshold - actual_value)
        else:
            raw_violation = max(0.0, actual_value - self.threshold)

        return raw_violation * self.penalty_weight


@dataclass
class EthicalSolution:
    """A CVRP plan together with its ethical evaluation."""

    solution_id: str
    routes: List[List[int]]
    vehicle_assignments: Dict[int, List[int]]
    traditional_metrics: Dict[str, float]
    ethical_metrics: Dict[str, float]
    stakeholder_utilities: Dict[int, float]
    constraint_violations: Dict[str, float]
    overall_ethical_score: float
    efficiency_score: float
    feasibility: bool


@dataclass
class EthicalFramework:
    """Configuration and performance record for the ethical framework."""

    framework_id: str
    stakeholders: List[Stakeholder]
    ethical_constraints: List[EthicalConstraint]
    ethical_principles: List[EthicalPrinciple]
    optimization_weights: Dict[str, float]

    def __post_init__(self) -> None:
        self.solution_history: List[EthicalSolution] = []
        self.performance_metrics: Dict[str, float] = {
            "ethical_satisfaction": 0.0,
            "stakeholder_balance": 0.0,
            "efficiency_retention": 0.0,
            "constraint_compliance": 0.0,
        }


print("Ethical framework data structures defined.")

In [ ]:
class StakeholderManagementSystem:
    """Create stakeholders and calculate their utility and balance."""

    def __init__(self) -> None:
        self.stakeholders: List[Stakeholder] = []
        self._initialize_stakeholders()

    def _initialize_stakeholders(self) -> None:
        self.stakeholders = [
            Stakeholder(
                stakeholder_id=1,
                stakeholder_type=StakeholderType.CUSTOMERS,
                name="Customer Group",
                interests=["service_reliability", "delivery_time", "price_affordability", "service_quality"],
                utility_weights={
                    "service_reliability": 0.4,
                    "delivery_time": -0.3,
                    "price_affordability": -0.2,
                    "service_quality": 0.1,
                },
                constraints=["fair_pricing", "service_accessibility", "quality_standards"],
                priority_level=0.9,
            ),
            Stakeholder(
                stakeholder_id=2,
                stakeholder_type=StakeholderType.DRIVERS,
                name="Driver Group",
                interests=["workload_balance", "fair_compensation", "working_conditions", "job_security"],
                utility_weights={
                    "workload_balance": 0.35,
                    "fair_compensation": 0.3,
                    "working_conditions": 0.25,
                    "job_security": 0.1,
                },
                constraints=["maximum_working_hours", "minimum_rest_time", "safety_standards"],
                priority_level=0.85,
            ),
            Stakeholder(
                stakeholder_id=3,
                stakeholder_type=StakeholderType.ENVIRONMENT,
                name="Environmental Group",
                interests=["carbon_emission_reduction", "energy_efficiency", "waste_minimization", "resource_conservation"],
                utility_weights={
                    "carbon_emission_reduction": 0.4,
                    "energy_efficiency": 0.3,
                    "waste_minimization": 0.2,
                    "resource_conservation": 0.1,
                },
                constraints=["emission_limits", "noise_pollution", "resource_usage"],
                priority_level=0.8,
            ),
            Stakeholder(
                stakeholder_id=4,
                stakeholder_type=StakeholderType.COMMUNITY,
                name="Community Group",
                interests=["local_employment", "traffic_reduction", "noise_control", "economic_development"],
                utility_weights={
                    "local_employment": 0.3,
                    "traffic_reduction": 0.25,
                    "noise_control": 0.25,
                    "economic_development": 0.2,
                },
                constraints=["traffic_congestion", "noise_levels", "local_hiring"],
                priority_level=0.75,
            ),
            Stakeholder(
                stakeholder_id=5,
                stakeholder_type=StakeholderType.COMPANY,
                name="Company Group",
                interests=["profitability", "operational_efficiency", "market_share", "brand_reputation"],
                utility_weights={
                    "profitability": 0.4,
                    "operational_efficiency": 0.3,
                    "market_share": 0.2,
                    "brand_reputation": 0.1,
                },
                constraints=["financial_sustainability", "regulatory_compliance", "competitive_pricing"],
                priority_level=0.8,
            ),
        ]

    def calculate_stakeholder_utilities(
        self, solution_metrics: Dict[str, float]
    ) -> Dict[int, float]:
        return {
            stakeholder.stakeholder_id: stakeholder.calculate_utility(solution_metrics)
            for stakeholder in self.stakeholders
        }

    def calculate_stakeholder_balance(self, utilities: Dict[int, float]) -> float:
        """Convert utility dispersion into a bounded balance indicator."""
        if not utilities:
            return 0.0

        values = np.asarray(list(utilities.values()), dtype=float)
        mean_utility = float(np.mean(values))
        if np.isclose(mean_utility, 0.0):
            return 0.0

        coefficient_of_variation = float(np.std(values) / abs(mean_utility))
        return float(max(0.0, 1.0 - coefficient_of_variation))

    def get_priority_weighted_utilities(
        self, utilities: Dict[int, float]
    ) -> Dict[int, float]:
        return {
            stakeholder.stakeholder_id: utilities[stakeholder.stakeholder_id]
            * stakeholder.priority_level
            for stakeholder in self.stakeholders
            if stakeholder.stakeholder_id in utilities
        }


print("Stakeholder management system defined.")

In [ ]:
class EthicalConstraintSystem:
    """Initialize constraints and summarize their violations."""

    def __init__(self) -> None:
        self.constraints: List[EthicalConstraint] = []
        self._initialize_constraints()

    def _initialize_constraints(self) -> None:
        self.constraints = [
            EthicalConstraint(
                constraint_id="workload_balance_001",
                constraint_type=ConstraintType.WORKLOAD_BALANCE,
                description="Require a workload-balance score of at least 0.80 (equivalent to load CV no greater than 0.20).",
                threshold=0.8,
                penalty_weight=0.3,
            ),
            EthicalConstraint(
                constraint_id="environmental_impact_001",
                constraint_type=ConstraintType.ENVIRONMENTAL_IMPACT,
                description="Limit modeled route emissions to 50 kg CO2.",
                threshold=50.0,
                penalty_weight=0.4,
            ),
            EthicalConstraint(
                constraint_id="service_equity_001",
                constraint_type=ConstraintType.SERVICE_EQUITY,
                description="Require the illustrative service-equity score to be at least 0.85.",
                threshold=0.85,
                penalty_weight=0.25,
            ),
            EthicalConstraint(
                constraint_id="driver_wellbeing_001",
                constraint_type=ConstraintType.DRIVER_WELLBEING,
                description="Limit modeled daily driver hours to 8 hours.",
                threshold=8.0,
                penalty_weight=0.35,
                is_hard_constraint=True,
            ),
            EthicalConstraint(
                constraint_id="community_impact_001",
                constraint_type=ConstraintType.COMMUNITY_IMPACT,
                description="Limit the simplified distance-based community-impact index to 0.15.",
                threshold=0.15,
                penalty_weight=0.2,
            ),
            EthicalConstraint(
                constraint_id="economic_viability_001",
                constraint_type=ConstraintType.ECONOMIC_VIABILITY,
                description="Require the illustrative profit-margin assumption to be at least 5%.",
                threshold=0.05,
                penalty_weight=0.3,
                is_hard_constraint=True,
            ),
        ]

    def evaluate_all_constraints(
        self, solution_metrics: Dict[str, float]
    ) -> Dict[str, float]:
        return {
            constraint.constraint_id: constraint.evaluate_violation(solution_metrics)
            for constraint in self.constraints
        }

    def calculate_compliance_score(self, violations: Dict[str, float]) -> float:
        """Return zero for any hard violation; otherwise score soft compliance."""
        hard_violation = any(
            constraint.is_hard_constraint
            and violations.get(constraint.constraint_id, 0.0) > 0.0
            for constraint in self.constraints
        )
        if hard_violation:
            return 0.0

        soft_constraints = [
            constraint for constraint in self.constraints
            if not constraint.is_hard_constraint
        ]
        if not soft_constraints:
            return 1.0

        total_violation = sum(
            violations.get(constraint.constraint_id, 0.0)
            for constraint in soft_constraints
        )
        return float(max(0.0, min(1.0, 1.0 - total_violation / len(soft_constraints))))

    def get_violation_summary(self, violations: Dict[str, float]) -> Dict[str, Any]:
        violated_constraints = []
        total_penalty = 0.0
        hard_violations = 0
        soft_violations = 0

        for constraint in self.constraints:
            violation = violations.get(constraint.constraint_id, 0.0)
            if violation <= 0.0:
                continue

            violated_constraints.append(
                {
                    "constraint_id": constraint.constraint_id,
                    "description": constraint.description,
                    "violation_amount": violation,
                    "is_hard": constraint.is_hard_constraint,
                }
            )
            total_penalty += violation
            if constraint.is_hard_constraint:
                hard_violations += 1
            else:
                soft_violations += 1

        return {
            "violated_constraints": violated_constraints,
            "total_violations": len(violated_constraints),
            "total_penalty": total_penalty,
            "hard_violations": hard_violations,
            "soft_violations": soft_violations,
        }


print("Ethical constraint system defined.")

## The optimization engine

The engine follows the stated sequence rather than solving a new CVRP from scratch. First it measures the supplied plan. It then attempts a single workload transfer between the most- and least-loaded vehicles when the difference exceeds 30% of the larger load; the receiving vehicle must have spare capacity. Finally, it applies bounded 2-opt search to each route. The workload transfer updates both the route and vehicle assignment so they remain consistent.

For vehicle loads \(L_v=\sum_{i\in R_v}q_i\), the load-balance score is \(\max(0,1-\sigma_L/\bar L)\), where \(\bar L\) and \(\sigma_L\) are the mean and population standard deviation of nonempty vehicle loads. Thus the score is between zero and one, and a score of at least 0.80 corresponds to a coefficient of variation no greater than 0.20. Empty vehicles are excluded from this calculation.

Modeled emissions are \(0.12D\) kg CO2, using the illustrative factor of 0.12 kg per kilometre. Driver hours use \(\ell(R_v)/30+0.5|R_v|\), with assumed average speed 30 km/h and 0.5 service hours per customer. These assumptions support the example's calculations; they are not measured operating data.

In [ ]:
class EthicalOptimizationEngine:
    """Post-process a supplied CVRP plan and calculate its ethical indicators."""

    def __init__(
        self,
        stakeholder_system: StakeholderManagementSystem,
        constraint_system: EthicalConstraintSystem,
    ) -> None:
        self.stakeholder_system = stakeholder_system
        self.constraint_system = constraint_system
        self.optimization_weights = {
            "efficiency": 0.3,
            "stakeholder_utility": 0.3,
            "ethical_compliance": 0.25,
            "environmental_impact": 0.15,
        }
        self.solution_history: List[EthicalSolution] = []

    def generate_ethical_solution(
        self,
        traditional_solution: Dict[str, Any],
        problem_context: Dict[str, Any],
    ) -> EthicalSolution:
        routes = [route.copy() for route in traditional_solution.get("routes", [])]
        assignments = {
            vehicle_id: customer_ids.copy()
            for vehicle_id, customer_ids
            in traditional_solution.get("vehicle_assignments", {}).items()
        }
        self._validate_plan(routes, assignments, problem_context)

        traditional_metrics = self._calculate_traditional_metrics(
            routes, assignments, problem_context
        )
        ethical_routes, ethical_assignments = self._apply_ethical_modifications(
            routes, assignments, problem_context
        )
        self._validate_plan(ethical_routes, ethical_assignments, problem_context)

        ethical_metrics = self._calculate_ethical_metrics(
            ethical_routes, ethical_assignments, problem_context
        )
        all_metrics = {**traditional_metrics, **ethical_metrics}
        stakeholder_utilities = self.stakeholder_system.calculate_stakeholder_utilities(
            all_metrics
        )
        constraint_violations = self.constraint_system.evaluate_all_constraints(
            all_metrics
        )
        overall_ethical_score = self._calculate_ethical_score(
            stakeholder_utilities, constraint_violations, ethical_metrics
        )
        efficiency_score = self._calculate_efficiency_retention(
            traditional_metrics, ethical_metrics
        )
        feasibility = self._check_feasibility(constraint_violations)

        solution = EthicalSolution(
            solution_id=f"ethical_{len(self.solution_history) + 1}",
            routes=ethical_routes,
            vehicle_assignments=ethical_assignments,
            traditional_metrics=traditional_metrics,
            ethical_metrics=ethical_metrics,
            stakeholder_utilities=stakeholder_utilities,
            constraint_violations=constraint_violations,
            overall_ethical_score=overall_ethical_score,
            efficiency_score=efficiency_score,
            feasibility=feasibility,
        )
        self.solution_history.append(solution)
        return solution

    def _validate_plan(
        self,
        routes: List[List[int]],
        vehicle_assignments: Dict[int, List[int]],
        problem_context: Dict[str, Any],
    ) -> None:
        """Check coverage, route-assignment agreement, and vehicle capacities."""
        customers = problem_context["customers"]
        vehicles = {vehicle["id"]: vehicle for vehicle in problem_context["vehicles"]}
        customer_by_id = {customer["id"]: customer for customer in customers}

        if len(routes) != len(vehicle_assignments):
            raise ValueError("Each route must correspond to one vehicle assignment.")
        if set(vehicle_assignments) - set(vehicles):
            raise ValueError("An assignment refers to a vehicle absent from the problem.")

        assigned_ids: List[int] = []
        for route, (vehicle_id, assigned_customers) in zip(
            routes, vehicle_assignments.items()
        ):
            if set(route) != set(assigned_customers):
                raise ValueError(
                    f"Route and assignment disagree for vehicle {vehicle_id}."
                )
            if any(customer_id not in customer_by_id for customer_id in route):
                raise ValueError("A route contains an unknown customer.")
            load = sum(customer_by_id[customer_id]["demand"] for customer_id in route)
            if load > vehicles[vehicle_id]["capacity"]:
                raise ValueError(f"Vehicle {vehicle_id} exceeds its capacity.")
            assigned_ids.extend(route)

        expected_ids = [customer["id"] for customer in customers]
        if sorted(assigned_ids) != sorted(expected_ids):
            raise ValueError("Every customer must be served exactly once.")

    def _calculate_traditional_metrics(
        self,
        routes: List[List[int]],
        vehicle_assignments: Dict[int, List[int]],
        problem_context: Dict[str, Any],
    ) -> Dict[str, float]:
        customers = problem_context["customers"]
        customer_by_id = {customer["id"]: customer for customer in customers}
        total_distance = sum(
            self._calculate_route_distance(route, customers) for route in routes
        )
        vehicle_loads = [
            sum(customer_by_id[customer_id]["demand"] for customer_id in customer_ids)
            for customer_ids in vehicle_assignments.values()
            if customer_ids
        ]
        load_balance = self._load_balance_score(vehicle_loads)
        total_demand = sum(customer["demand"] for customer in customers)
        vehicles_used = sum(bool(customer_ids) for customer_ids in vehicle_assignments.values())

        return {
            "total_distance": total_distance,
            "vehicles_used": float(vehicles_used),
            "total_demand": float(total_demand),
            "load_balance": load_balance,
            "efficiency_score": max(0.0, 1.0 - total_distance / 1000.0),
        }

    @staticmethod
    def _load_balance_score(vehicle_loads: List[float]) -> float:
        if not vehicle_loads:
            return 1.0
        mean_load = float(np.mean(vehicle_loads))
        if np.isclose(mean_load, 0.0):
            return 1.0
        coefficient_of_variation = float(np.std(vehicle_loads) / mean_load)
        return float(max(0.0, 1.0 - coefficient_of_variation))

    def _calculate_route_distance(
        self, route: List[int], customers: List[Dict[str, Any]]
    ) -> float:
        customer_by_id = {customer["id"]: customer for customer in customers}
        depot = (0.0, 0.0)
        total_distance = 0.0
        current_location = depot

        for customer_id in route:
            if customer_id not in customer_by_id:
                raise ValueError(f"Unknown customer ID: {customer_id}")
            location = customer_by_id[customer_id]["location"]
            total_distance += float(
                np.hypot(location[0] - current_location[0], location[1] - current_location[1])
            )
            current_location = location

        total_distance += float(
            np.hypot(depot[0] - current_location[0], depot[1] - current_location[1])
        )
        return total_distance

    def _apply_ethical_modifications(
        self,
        routes: List[List[int]],
        vehicle_assignments: Dict[int, List[int]],
        problem_context: Dict[str, Any],
    ) -> Tuple[List[List[int]], Dict[int, List[int]]]:
        balanced_routes, balanced_assignments = self._balance_workload(
            routes, vehicle_assignments, problem_context
        )
        optimized_routes = self._optimize_for_environment(
            balanced_routes, problem_context
        )
        return optimized_routes, balanced_assignments

    def _balance_workload(
        self,
        routes: List[List[int]],
        vehicle_assignments: Dict[int, List[int]],
        problem_context: Dict[str, Any],
    ) -> Tuple[List[List[int]], Dict[int, List[int]]]:
        """Attempt one capacity-feasible transfer between the most uneven loads."""
        customers = problem_context["customers"]
        customer_by_id = {customer["id"]: customer for customer in customers}
        vehicles = {vehicle["id"]: vehicle for vehicle in problem_context["vehicles"]}
        balanced_routes = [route.copy() for route in routes]
        balanced_assignments = {
            vehicle_id: customer_ids.copy()
            for vehicle_id, customer_ids in vehicle_assignments.items()
        }
        vehicle_ids = list(balanced_assignments)
        loads = {
            vehicle_id: sum(
                customer_by_id[customer_id]["demand"]
                for customer_id in balanced_assignments[vehicle_id]
            )
            for vehicle_id in vehicle_ids
        }

        if len(loads) < 2:
            return balanced_routes, balanced_assignments

        overloaded_vehicle = max(loads, key=loads.get)
        underloaded_vehicle = min(loads, key=loads.get)
        max_load = loads[overloaded_vehicle]
        min_load = loads[underloaded_vehicle]
        if max_load <= 0 or max_load - min_load <= max_load * 0.3:
            return balanced_routes, balanced_assignments

        overloaded_index = vehicle_ids.index(overloaded_vehicle)
        underloaded_index = vehicle_ids.index(underloaded_vehicle)
        receiver_capacity = vehicles[underloaded_vehicle]["capacity"]

        for customer_id in balanced_assignments[overloaded_vehicle]:
            demand = customer_by_id[customer_id]["demand"]
            if min_load + demand <= receiver_capacity:
                balanced_assignments[overloaded_vehicle].remove(customer_id)
                balanced_assignments[underloaded_vehicle].append(customer_id)
                balanced_routes[overloaded_index].remove(customer_id)
                balanced_routes[underloaded_index].append(customer_id)
                break

        return balanced_routes, balanced_assignments

    def _optimize_for_environment(
        self, routes: List[List[int]], problem_context: Dict[str, Any]
    ) -> List[List[int]]:
        customers = problem_context["customers"]
        return [
            self._two_opt_optimization(route, customers) if len(route) > 1 else route.copy()
            for route in routes
        ]

    def _two_opt_optimization(
        self, route: List[int], customers: List[Dict[str, Any]]
    ) -> List[int]:
        """Search bounded 2-opt reversals and retain the shortest route found."""
        if len(route) <= 2:
            return route.copy()

        best_route = route.copy()
        best_distance = self._calculate_route_distance(best_route, customers)
        current_route = best_route.copy()
        max_iterations = min(100, len(route) * 10)

        for _ in range(max_iterations):
            improved = False
            for i in range(len(current_route) - 1):
                for j in range(i + 2, len(current_route)):
                    candidate = (
                        current_route[: i + 1]
                        + current_route[i + 1 : j + 1][::-1]
                        + current_route[j + 1 :]
                    )
                    candidate_distance = self._calculate_route_distance(
                        candidate, customers
                    )
                    if candidate_distance < best_distance:
                        best_route = candidate
                        best_distance = candidate_distance
                        improved = True
            if not improved:
                break
            current_route = best_route.copy()

        return best_route

    def _calculate_ethical_metrics(
        self,
        routes: List[List[int]],
        vehicle_assignments: Dict[int, List[int]],
        problem_context: Dict[str, Any],
    ) -> Dict[str, float]:
        customers = problem_context["customers"]
        customer_by_id = {customer["id"]: customer for customer in customers}
        total_distance = sum(
            self._calculate_route_distance(route, customers) for route in routes
        )
        carbon_emissions = total_distance * 0.12

        vehicle_loads = [
            sum(customer_by_id[customer_id]["demand"] for customer_id in customer_ids)
            for customer_ids in vehicle_assignments.values()
            if customer_ids
        ]
        workload_balance = self._load_balance_score(vehicle_loads)

        # These scenario inputs are illustrative because measured values are unavailable.
        service_equity = 0.9
        economic_viability = 0.12

        route_hours = [
            self._calculate_route_distance(route, customers) / 30.0 + 0.5 * len(route)
            for route in routes
        ]
        driver_hours = max(route_hours, default=0.0)
        driver_wellbeing = max(0.0, 1.0 - driver_hours / 8.0)
        community_impact = total_distance * 0.01

        return {
            "total_distance": total_distance,
            "workload_balance": workload_balance,
            "environmental_impact": carbon_emissions,
            "carbon_emission_reduction": max(0.0, 1.0 - carbon_emissions / 100.0),
            "service_equity": service_equity,
            "driver_hours": driver_hours,
            "driver_wellbeing": driver_wellbeing,
            "community_impact": community_impact,
            "traffic_reduction": max(0.0, 1.0 - min(1.0, community_impact)),
            "economic_viability": economic_viability,
        }

    def _calculate_ethical_score(
        self,
        stakeholder_utilities: Dict[int, float],
        constraint_violations: Dict[str, float],
        ethical_metrics: Dict[str, float],
    ) -> float:
        weighted_utilities = self.stakeholder_system.get_priority_weighted_utilities(
            stakeholder_utilities
        )
        utility_score = (
            float(np.mean(list(weighted_utilities.values())))
            if weighted_utilities
            else 0.0
        )
        compliance_score = self.constraint_system.calculate_compliance_score(
            constraint_violations
        )
        metric_score = float(
            np.mean(
                [
                    ethical_metrics["workload_balance"],
                    ethical_metrics["service_equity"],
                    ethical_metrics["driver_wellbeing"],
                    1.0 - min(1.0, ethical_metrics["environmental_impact"] / 100.0),
                    1.0 - min(1.0, ethical_metrics["community_impact"] / 0.5),
                    min(1.0, ethical_metrics["economic_viability"] / 0.2),
                ]
            )
        )
        overall_score = utility_score * 0.4 + compliance_score * 0.35 + metric_score * 0.25
        return float(max(0.0, min(1.0, overall_score)))

    @staticmethod
    def _calculate_efficiency_retention(
        traditional_metrics: Dict[str, float], ethical_metrics: Dict[str, float]
    ) -> float:
        """Compare calculated distances; a shorter modified plan retains 100%."""
        baseline_distance = traditional_metrics["total_distance"]
        modified_distance = ethical_metrics["total_distance"]
        if modified_distance <= 0.0:
            return 1.0 if baseline_distance <= 0.0 else 0.0
        return float(max(0.0, min(1.0, baseline_distance / modified_distance)))

    def _check_feasibility(self, constraint_violations: Dict[str, float]) -> bool:
        return not any(
            constraint.is_hard_constraint
            and constraint_violations.get(constraint.constraint_id, 0.0) > 0.0
            for constraint in self.constraint_system.constraints
        )


print("Ethical optimization engine defined.")

## The CVRP evaluation system

This coordinating class connects stakeholder management, constraint evaluation, and the post-processing engine. The order matters: plan validation protects the CVRP structure, and all reported scores are calculated from the same final routes and assignments. The original plan's stated reference distance and cost are retained as descriptive input fields; route distance used for evaluation is calculated from the customer coordinates and route sequence.

In [ ]:
class EthicalCVRPSystem:
    """Coordinate plan evaluation, performance tracking, and reporting."""

    def __init__(self) -> None:
        self.stakeholder_system = StakeholderManagementSystem()
        self.constraint_system = EthicalConstraintSystem()
        self.optimization_engine = EthicalOptimizationEngine(
            self.stakeholder_system, self.constraint_system
        )
        self.ethical_framework = EthicalFramework(
            framework_id="ethical_cvrp_001",
            stakeholders=self.stakeholder_system.stakeholders,
            ethical_constraints=self.constraint_system.constraints,
            ethical_principles=list(EthicalPrinciple),
            optimization_weights=self.optimization_engine.optimization_weights,
        )
        self.solutions: List[EthicalSolution] = []
        self.performance_history: List[Dict[str, float]] = []

    def solve_ethical_cvrp(
        self,
        traditional_solution: Dict[str, Any],
        problem_context: Dict[str, Any],
    ) -> EthicalSolution:
        print("Generating and evaluating the ethical CVRP plan...")
        solution = self.optimization_engine.generate_ethical_solution(
            traditional_solution, problem_context
        )
        self.solutions.append(solution)
        self._update_performance_metrics(solution)
        self.ethical_framework.solution_history.append(solution)
        return solution

    def _update_performance_metrics(self, solution: EthicalSolution) -> None:
        current = {
            "ethical_satisfaction": solution.overall_ethical_score,
            "stakeholder_balance": self.stakeholder_system.calculate_stakeholder_balance(
                solution.stakeholder_utilities
            ),
            "efficiency_retention": solution.efficiency_score,
            "constraint_compliance": self.constraint_system.calculate_compliance_score(
                solution.constraint_violations
            ),
        }
        self.performance_history.append(current)
        self.ethical_framework.performance_metrics = current.copy()

    def analyze_solution(self, solution: EthicalSolution) -> Dict[str, Any]:
        stakeholder_analysis = {}
        for stakeholder in self.stakeholder_system.stakeholders:
            utility = solution.stakeholder_utilities.get(stakeholder.stakeholder_id, 0.0)
            stakeholder_analysis[stakeholder.name] = {
                "utility_score": utility,
                "priority_level": stakeholder.priority_level,
                "key_interests": stakeholder.interests[:2],
                "satisfaction_status": (
                    "High" if utility > 0.7 else "Medium" if utility > 0.3 else "Low"
                ),
            }

        constraint_analysis = self.constraint_system.get_violation_summary(
            solution.constraint_violations
        )
        metrics = solution.ethical_metrics
        ethical_metrics_analysis = {
            "workload_balance": {
                "score": metrics["workload_balance"],
                "status": "Strong" if metrics["workload_balance"] > 0.8 else "Needs review",
            },
            "environmental_impact": {
                "emissions_kg": metrics["environmental_impact"],
                "status": "Within illustrative threshold"
                if metrics["environmental_impact"] <= 50.0
                else "Above illustrative threshold",
            },
            "service_equity": {
                "score": metrics["service_equity"],
                "status": "Illustrative input; not measured",
            },
            "economic_viability": {
                "profit_margin": metrics["economic_viability"],
                "status": "Illustrative input; not measured",
            },
        }
        utilities = list(solution.stakeholder_utilities.values())
        utility_range = max(utilities) - min(utilities) if utilities else 0.0
        average_utility = float(np.mean(utilities)) if utilities else 0.0
        balance_score = self.stakeholder_system.calculate_stakeholder_balance(
            solution.stakeholder_utilities
        )
        trade_off_analysis = {
            "efficiency_vs_ethics": {
                "efficiency_retained": solution.efficiency_score,
                "ethical_score": solution.overall_ethical_score,
                "balance_status": (
                    "Both indicators are high"
                    if solution.efficiency_score > 0.8
                    and solution.overall_ethical_score > 0.8
                    else "Ethical score exceeds efficiency retention"
                    if solution.overall_ethical_score > solution.efficiency_score
                    else "Efficiency retention equals or exceeds ethical score"
                ),
            },
            "stakeholder_satisfaction": {
                "average_utility": average_utility,
                "utility_range": utility_range,
                "balance_score": balance_score,
            },
        }
        recommendation = (
            "Review the plan and assumptions before operational use"
            if solution.feasibility and solution.overall_ethical_score > 0.7
            else "Review constraint violations and model assumptions"
            if solution.feasibility
            else "Do not accept: at least one hard ethical constraint is violated"
        )

        return {
            "stakeholder_analysis": stakeholder_analysis,
            "constraint_analysis": constraint_analysis,
            "ethical_metrics_analysis": ethical_metrics_analysis,
            "trade_off_analysis": trade_off_analysis,
            "overall_assessment": {
                "ethical_score": solution.overall_ethical_score,
                "feasibility": solution.feasibility,
                "recommendation": recommendation,
            },
        }

    def generate_ethical_report(self, solution: EthicalSolution) -> str:
        analysis = self.analyze_solution(solution)
        lines = [
            "ETHICAL CVRP SOLUTION EVALUATION REPORT",
            "=" * 42,
            f"Solution ID: {solution.solution_id}",
            f"Overall ethical score: {solution.overall_ethical_score:.1%}",
            f"Efficiency retention: {solution.efficiency_score:.1%}",
            f"Feasibility: {'FEASIBLE' if solution.feasibility else 'INFEASIBLE'}",
            f"Calculated route distance: {solution.ethical_metrics['total_distance']:.2f} km",
            "",
            "STAKEHOLDER UTILITIES",
        ]
        for name, data in analysis["stakeholder_analysis"].items():
            lines.append(
                f"- {name}: {data['utility_score']:.3f} "
                f"(priority {data['priority_level']:.2f}; "
                f"{data['satisfaction_status']} by the illustrative thresholds)"
            )

        constraint_summary = analysis["constraint_analysis"]
        lines.extend(
            [
                "",
                "CONSTRAINT EVALUATION",
                f"- Violated constraints: {constraint_summary['total_violations']}",
                f"- Hard violations: {constraint_summary['hard_violations']}",
                f"- Soft violations: {constraint_summary['soft_violations']}",
                f"- Sum of weighted violations: {constraint_summary['total_penalty']:.3f}",
                "",
                "MODEL LIMITATIONS",
                "- Several stakeholder metrics are unavailable and contribute zero to utility.",
                "- Service equity and profit margin are illustrative inputs.",
                "- Driver hours and community impact use simplified assumptions.",
                "- Scores describe this model and plan; they are not empirical impact estimates.",
                "",
                f"Interpretation: {analysis['overall_assessment']['recommendation']}",
            ]
        )
        return "\n".join(lines)


print("Ethical CVRP evaluation system defined.")

## Configure the ethical framework

The configuration makes the five stakeholder groups, six ethical constraints, and six principles explicit. A hard constraint determines feasibility; a soft constraint contributes to the compliance score. Penalty weights scale violation amounts, so violations with different units should not be interpreted as directly comparable measurements.

In [ ]:
ethical_system = EthicalCVRPSystem()

print("ETHICAL FRAMEWORK CONFIGURATION")
print("=" * 45)
print(f"Stakeholder groups: {len(ethical_system.stakeholder_system.stakeholders)}")
print(f"Ethical constraints: {len(ethical_system.constraint_system.constraints)}")
print(f"Ethical principles: {len(ethical_system.ethical_framework.ethical_principles)}")
print("Evaluation sequence: workload transfer, 2-opt route improvement, ethical scoring")

print("\nStakeholder groups:")
for stakeholder in ethical_system.stakeholder_system.stakeholders:
    print(
        f"- {stakeholder.name} ({stakeholder.stakeholder_type.value}); "
        f"priority={stakeholder.priority_level:.2f}; "
        f"key interests={', '.join(stakeholder.interests[:2])}"
    )

print("\nEthical constraints:")
for constraint in ethical_system.constraint_system.constraints:
    kind = "HARD" if constraint.is_hard_constraint else "SOFT"
    print(
        f"- {constraint.description} [{kind}; threshold={constraint.threshold}; "
        f"penalty weight={constraint.penalty_weight:.2f}]"
    )

print("\nEthical principles represented:")
for principle in ethical_system.ethical_framework.ethical_principles:
    print(f"- {principle.value.title()}")

## Define the sample CVRP

The instance contains ten customers and four vehicles. Three vehicles are used by the supplied plan; the fourth is available but initially unused. The routes are grouped geographically, and the input includes a reference distance and cost. The reference distance is preserved as supplied metadata, while the model calculates route distance directly from the coordinates. The two distance values should not be assumed to agree.

In [ ]:
def create_sample_cvrp_problem() -> Dict[str, Any]:
    """Return the sample customers, vehicles, plan, and scenario context."""
    customers = [
        {"id": 1, "location": (5, 10), "demand": 3, "priority": "high", "area": "urban"},
        {"id": 2, "location": (15, 5), "demand": 2, "priority": "medium", "area": "suburban"},
        {"id": 3, "location": (10, 15), "demand": 4, "priority": "high", "area": "urban"},
        {"id": 4, "location": (20, 10), "demand": 2, "priority": "low", "area": "rural"},
        {"id": 5, "location": (8, 12), "demand": 3, "priority": "medium", "area": "urban"},
        {"id": 6, "location": (12, 8), "demand": 2, "priority": "high", "area": "suburban"},
        {"id": 7, "location": (18, 12), "demand": 3, "priority": "medium", "area": "rural"},
        {"id": 8, "location": (6, 8), "demand": 4, "priority": "high", "area": "urban"},
        {"id": 9, "location": (14, 18), "demand": 2, "priority": "low", "area": "rural"},
        {"id": 10, "location": (16, 6), "demand": 3, "priority": "medium", "area": "suburban"},
    ]
    vehicles = [
        {"id": 1, "capacity": 15, "location": (0, 0), "driver_experience": "senior", "fuel_type": "electric"},
        {"id": 2, "capacity": 12, "location": (0, 0), "driver_experience": "junior", "fuel_type": "gasoline"},
        {"id": 3, "capacity": 18, "location": (0, 0), "driver_experience": "senior", "fuel_type": "hybrid"},
        {"id": 4, "capacity": 10, "location": (0, 0), "driver_experience": "junior", "fuel_type": "electric"},
    ]
    traditional_routes = [
        [1, 3, 5, 8],
        [2, 6, 10],
        [4, 7, 9],
    ]
    traditional_assignments = {
        1: [1, 3, 5, 8],
        2: [2, 6, 10],
        3: [4, 7, 9],
    }
    traditional_solution = {
        "routes": traditional_routes,
        "vehicle_assignments": traditional_assignments,
        "total_distance": 85.2,
        "total_cost": 425.0,
        "vehicles_used": 3,
    }
    problem_context = {
        "customers": customers,
        "vehicles": vehicles,
        "service_area": "mixed_urban_suburban_rural",
        "environmental_regulations": "strict",
        "community_requirements": ["low_noise", "local_hiring", "traffic_reduction"],
        "company_goals": ["profitability", "sustainability", "brand_reputation"],
    }
    return {
        "traditional_solution": traditional_solution,
        "problem_context": problem_context,
        "description": "Mixed urban-rural CVRP with 10 customers and 4 vehicles",
    }


sample_problem = create_sample_cvrp_problem()
reference_solution = sample_problem["traditional_solution"]
problem_context = sample_problem["problem_context"]
calculated_reference_distance = sum(
    ethical_system.optimization_engine._calculate_route_distance(
        route, problem_context["customers"]
    )
    for route in reference_solution["routes"]
)

print("Sample CVRP problem:")
print(f"- Description: {sample_problem['description']}")
print(f"- Customers: {len(problem_context['customers'])}")
print(f"- Available vehicles: {len(problem_context['vehicles'])}")
print(f"- Service area: {problem_context['service_area']}")
print(f"- Environmental regulation setting: {problem_context['environmental_regulations']}")
print(f"- Supplied reference distance: {reference_solution['total_distance']:.1f} km")
print(f"- Distance calculated from the supplied routes: {calculated_reference_distance:.2f} km")
print(f"- Supplied reference cost: ${reference_solution['total_cost']:.2f}")

## Generate and inspect the ethical plan

The run below applies the workload transfer and route-level 2-opt heuristic, then evaluates the plan. The distance comparison uses the route geometry for both plans; the separate supplied reference distance is not substituted for the calculated baseline. Vehicle capacities, customer coverage, and consistency between routes and assignments are checked before and after post-processing.

In [ ]:
ethical_solution = ethical_system.solve_ethical_cvrp(
    sample_problem["traditional_solution"],
    sample_problem["problem_context"],
)

traditional_distance = ethical_solution.traditional_metrics["total_distance"]
ethical_distance = ethical_solution.ethical_metrics["total_distance"]
distance_change = (
    0.0
    if np.isclose(traditional_distance, 0.0)
    else (ethical_distance - traditional_distance) / traditional_distance * 100.0
)

assert np.isclose(
    sum(ethical_solution.ethical_metrics["total_distance"] for _ in [0]),
    ethical_distance,
)
assert set(ethical_solution.vehicle_assignments) == {1, 2, 3}
assert all(
    set(route) == set(ethical_solution.vehicle_assignments[vehicle_id])
    for route, vehicle_id in zip(
        ethical_solution.routes, ethical_solution.vehicle_assignments
    )
)

print("ETHICAL CVRP EVALUATION")
print("=" * 45)
print(f"Solution ID: {ethical_solution.solution_id}")
print(f"Overall ethical score: {ethical_solution.overall_ethical_score:.1%}")
print(f"Efficiency retention: {ethical_solution.efficiency_score:.1%}")
print(f"Feasibility under the defined hard constraints: {ethical_solution.feasibility}")
print("\nCalculated route-distance comparison:")
print(f"- Initial plan: {traditional_distance:.2f} km")
print(f"- Evaluated plan: {ethical_distance:.2f} km")
print(f"- Change: {distance_change:+.2f}%")
print(f"- Vehicles used: {int(ethical_solution.traditional_metrics['vehicles_used'])}")
print(f"- Initial-plan load-balance score: {ethical_solution.traditional_metrics['load_balance']:.3f}")
print("\nEthical indicators:")
print(f"- Workload balance: {ethical_solution.ethical_metrics['workload_balance']:.3f}")
print(f"- Modeled emissions: {ethical_solution.ethical_metrics['environmental_impact']:.2f} kg CO2")
print(f"- Illustrative service equity: {ethical_solution.ethical_metrics['service_equity']:.2f}")
print(f"- Modeled maximum driver hours: {ethical_solution.ethical_metrics['driver_hours']:.2f}")
print(f"- Simplified community-impact index: {ethical_solution.ethical_metrics['community_impact']:.3f}")
print(f"- Illustrative profit margin: {ethical_solution.ethical_metrics['economic_viability']:.1%}")

print("\nStakeholder utilities (unavailable metrics contribute zero):")
for stakeholder in ethical_system.stakeholder_system.stakeholders:
    utility = ethical_solution.stakeholder_utilities[stakeholder.stakeholder_id]
    print(f"- {stakeholder.name}: {utility:.3f}")

constraint_summary = ethical_system.constraint_system.get_violation_summary(
    ethical_solution.constraint_violations
)
compliance_score = ethical_system.constraint_system.calculate_compliance_score(
    ethical_solution.constraint_violations
)
print("\nConstraint evaluation:")
print(f"- Violated constraints: {constraint_summary['total_violations']}")
print(f"- Hard violations: {constraint_summary['hard_violations']}")
print(f"- Soft violations: {constraint_summary['soft_violations']}")
print(f"- Compliance score: {compliance_score:.1%}")

## Comprehensive ethical analysis

The analysis separates utility scores, constraint violations, and performance indicators because they answer different questions. Utility depends on stakeholder weights and available metrics; compliance depends on thresholds and hard-versus-soft status; route distance is calculated from the plan. The sample's unavailable baseline data do not support claims about measured environmental reductions or social-equity improvements, so the analysis reports the model's values and assumptions rather than asserting such improvements.

In [ ]:
def analyze_ethical_performance(
    ethical_system: EthicalCVRPSystem,
    ethical_solution: EthicalSolution,
    sample_problem: Dict[str, Any],
) -> Dict[str, Any]:
    """Print a structured analysis and return the underlying analysis data."""
    analysis = ethical_system.analyze_solution(ethical_solution)

    print("COMPREHENSIVE ETHICAL ANALYSIS")
    print("=" * 45)
    print("\nStakeholder utilities:")
    for name, data in analysis["stakeholder_analysis"].items():
        interests = ", ".join(data["key_interests"])
        print(
            f"- {name}: utility={data['utility_score']:.3f}, "
            f"priority={data['priority_level']:.2f}, "
            f"status={data['satisfaction_status']}; key interests: {interests}"
        )

    constraint_analysis = analysis["constraint_analysis"]
    total_constraints = len(ethical_system.constraint_system.constraints)
    satisfied_count = total_constraints - constraint_analysis["total_violations"]
    satisfaction_fraction = satisfied_count / total_constraints if total_constraints else 1.0
    print("\nConstraint evaluation:")
    print(f"- Total constraints: {total_constraints}")
    print(f"- Constraints with no recorded violation: {satisfied_count}")
    print(
        f"- Unweighted count-based satisfaction fraction: "
        f"{satisfaction_fraction:.1%}"
    )
    print(f"- Sum of weighted violations: {constraint_analysis['total_penalty']:.3f}")
    if constraint_analysis["violated_constraints"]:
        print("- Violations:")
        for violation in constraint_analysis["violated_constraints"]:
            kind = "HARD" if violation["is_hard"] else "SOFT"
            print(
                f"  - {violation['description']} ({kind}); "
                f"weighted violation={violation['violation_amount']:.3f}"
            )

    print("\nEthical indicators:")
    for metric_name, data in analysis["ethical_metrics_analysis"].items():
        if metric_name == "environmental_impact":
            print(f"- Modeled emissions: {data['emissions_kg']:.2f} kg CO2; {data['status']}")
        elif metric_name == "economic_viability":
            print(f"- Profit margin assumption: {data['profit_margin']:.1%}; {data['status']}")
        else:
            print(f"- {metric_name.replace('_', ' ').title()}: {data['score']:.3f}; {data['status']}")

    trade_off = analysis["trade_off_analysis"]
    print("\nTrade-off indicators:")
    print(f"- Calculated-distance efficiency retention: {ethical_solution.efficiency_score:.1%}")
    print(f"- Ethical score: {ethical_solution.overall_ethical_score:.1%}")
    print(f"- Stakeholder utility balance indicator: {trade_off['stakeholder_satisfaction']['balance_score']:.3f}")
    print(
        f"- Average stakeholder utility: "
        f"{trade_off['stakeholder_satisfaction']['average_utility']:.3f}"
    )
    print("\nInterpretation:")
    print("- These scores follow the stated illustrative weights, thresholds, and assumptions.")
    print("- Missing stakeholder metrics are not evidence of satisfaction or dissatisfaction.")
    print("- The sample does not support empirical percentage claims about environmental or social impact.")
    print(f"- Supplied reference distance: {sample_problem['traditional_solution']['total_distance']:.1f} km.")
    print(
        f"- Geometry-based initial-plan distance: "
        f"{ethical_solution.traditional_metrics['total_distance']:.2f} km."
    )
    return analysis


ethical_analysis = analyze_ethical_performance(
    ethical_system, ethical_solution, sample_problem
)

## Evaluation report and performance summary

The report is generated from the evaluated solution object. Its feasibility label concerns the declared hard ethical constraints only; CVRP coverage and capacity are separately enforced by plan validation. The report also distinguishes illustrative inputs and simplified indicators from observed data.

In [ ]:
ethical_report = ethical_system.generate_ethical_report(ethical_solution)
print(ethical_report)

print("\nFRAMEWORK PERFORMANCE SUMMARY")
print("=" * 45)
print(f"Overall ethical score: {ethical_solution.overall_ethical_score:.1%}")
print(f"Efficiency retention: {ethical_solution.efficiency_score:.1%}")
print(
    "Stakeholder balance indicator: "
    f"{ethical_system.stakeholder_system.calculate_stakeholder_balance(ethical_solution.stakeholder_utilities):.3f}"
)
print(f"Constraint compliance score: {compliance_score:.1%}")
print("\nFramework capabilities represented in this example:")
print("- Stakeholder utility scoring with explicit priorities")
print("- Capacity-aware workload transfer and route-level 2-opt search")
print("- Hard and soft constraint evaluation")
print("- Transparent reporting of assumptions and weighted violations")

## Visualize the evaluated plan

The panels show stakeholder utilities, selected ethical indicators, constraint status, and a compact summary. Scores are labeled with their units or interpretation where applicable. A stakeholder utility can be negative or exceed one because it is a weighted sum; the chart therefore does not impose a misleading zero-to-one scale on utilities.

In [ ]:
def visualize_ethical_framework(
    ethical_system: EthicalCVRPSystem,
    ethical_solution: EthicalSolution,
    ethical_analysis: Dict[str, Any],
) -> None:
    """Plot stakeholder utilities, indicators, constraints, and a summary."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 11))
    ax1, ax2, ax3, ax4 = axes.flatten()

    stakeholders = ethical_system.stakeholder_system.stakeholders
    names = [stakeholder.name.replace(" Group", "") for stakeholder in stakeholders]
    utilities = [
        ethical_solution.stakeholder_utilities[stakeholder.stakeholder_id]
        for stakeholder in stakeholders
    ]
    priorities = [stakeholder.priority_level for stakeholder in stakeholders]
    positions = np.arange(len(names))
    bars = ax1.bar(positions, utilities, color="steelblue", edgecolor="navy", alpha=0.8)
    ax1.axhline(0.0, color="black", linewidth=0.8)
    ax1.set_title("Stakeholder Utility")
    ax1.set_xlabel("Stakeholder")
    ax1.set_ylabel("Weighted utility (unitless)")
    ax1.set_xticks(positions)
    ax1.set_xticklabels(names, rotation=25, ha="right")
    ax1.grid(axis="y", alpha=0.25)
    for bar, priority in zip(bars, priorities):
        offset = 0.02 if bar.get_height() >= 0 else -0.04
        ax1.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + offset,
            f"Priority {priority:.2f}",
            ha="center",
            va="bottom" if bar.get_height() >= 0 else "top",
            fontsize=8,
        )

    metrics = ethical_solution.ethical_metrics
    metric_names = ["Workload balance", "Service equity", "Driver wellbeing", "Profit margin"]
    metric_values = [
        metrics["workload_balance"],
        metrics["service_equity"],
        metrics["driver_wellbeing"],
        metrics["economic_viability"],
    ]
    metric_positions = np.arange(len(metric_names))
    ax2.bar(metric_positions, metric_values, color="seagreen", edgecolor="darkgreen", alpha=0.8)
    ax2.set_title("Selected Ethical Indicators")
    ax2.set_ylabel("Score or margin (unitless)")
    ax2.set_xticks(metric_positions)
    ax2.set_xticklabels(metric_names, rotation=25, ha="right")
    ax2.set_ylim(0.0, 1.0)
    ax2.grid(axis="y", alpha=0.25)
    for position, value in zip(metric_positions, metric_values):
        ax2.text(position, min(value + 0.02, 0.98), f"{value:.2f}", ha="center", fontsize=8)

    constraints = ethical_system.constraint_system.constraints
    constraint_names = [
        constraint.constraint_type.value.replace("_", " ").title()
        for constraint in constraints
    ]
    compliance_values = [
        1.0 if ethical_solution.constraint_violations.get(constraint.constraint_id, 0.0) <= 0.0 else 0.0
        for constraint in constraints
    ]
    colors = ["seagreen" if value == 1.0 else "firebrick" for value in compliance_values]
    constraint_positions = np.arange(len(constraint_names))
    ax3.barh(constraint_positions, compliance_values, color=colors, alpha=0.8)
    ax3.set_title("Constraint Threshold Status")
    ax3.set_xlabel("Satisfied (1) or violated (0)")
    ax3.set_xlim(0.0, 1.15)
    ax3.set_yticks(constraint_positions)
    ax3.set_yticklabels(constraint_names)
    ax3.grid(axis="x", alpha=0.25)
    ax3.legend(
        handles=[
            plt.Rectangle((0, 0), 1, 1, color="seagreen", label="Satisfied"),
            plt.Rectangle((0, 0), 1, 1, color="firebrick", label="Violated"),
        ],
        loc="lower right",
    )

    ax4.axis("off")
    summary = (
        "Evaluated plan summary\n"
        "─────────────────────\n"
        f"Ethical score: {ethical_solution.overall_ethical_score:.1%}\n"
        f"Efficiency retention: {ethical_solution.efficiency_score:.1%}\n"
        f"Feasible under hard ethical constraints: {ethical_solution.feasibility}\n"
        f"Calculated distance: {metrics['total_distance']:.2f} km\n"
        f"Modeled emissions: {metrics['environmental_impact']:.2f} kg CO2\n"
        f"Modeled maximum driver hours: {metrics['driver_hours']:.2f}\n"
        f"Soft/hard violations: "
        f"{ethical_analysis['constraint_analysis']['soft_violations']}/"
        f"{ethical_analysis['constraint_analysis']['hard_violations']}\n\n"
        "Interpretation notes\n"
        "• Utilities depend on available metrics and weights.\n"
        "• Equity and profit margin are illustrative inputs.\n"
        "• Driver hours and community impact are simplified estimates.\n"
        "• This is a heuristic evaluation, not a global CVRP optimum."
    )
    ax4.text(
        0.03,
        0.97,
        summary,
        fontsize=10,
        verticalalignment="top",
        fontfamily="monospace",
        bbox={"boxstyle": "round", "facecolor": "whitesmoke", "alpha": 0.9},
    )

    fig.suptitle("Ethical CVRP Plan Evaluation", fontsize=15)
    fig.tight_layout()
    plt.show()


visualize_ethical_framework(ethical_system, ethical_solution, ethical_analysis)

## Why this tier matters

An efficiency-only route plan does not, by itself, describe who receives benefits, how work is distributed, or which social and environmental effects are considered. This framework makes selected values and constraints visible alongside the routing plan. Its practical value lies in transparent measurement and discussion of trade-offs—not in claiming that a weighted score settles ethical questions.

### Strengths

- Represents several stakeholder groups and exposes their utility weights and priority levels.
- Distinguishes hard constraints, which determine feasibility, from soft constraints, which affect compliance scoring.
- Includes workload balancing and route-distance improvement in a clearly ordered post-processing sequence.
- Produces a traceable summary of metrics, violations, and modeling assumptions.

### Limitations

- Utility weights, thresholds, and score normalizations involve value judgments and require stakeholder validation.
- Missing or placeholder metrics limit what stakeholder utilities and ethical scores can establish.
- The workload step attempts only one transfer; the 2-opt step is a bounded local search. Neither guarantees a globally optimal CVRP plan.
- Euclidean distances and simplified emissions, hours, and community indicators omit important operational conditions.
- A single aggregate score can conceal conflicts among stakeholders and should be read alongside its components.

### Appropriate use

This type of framework can support educational analysis, preliminary policy discussion, and transparent comparison of explicitly modeled priorities. Operational use requires validated route data, measured stakeholder outcomes, justified thresholds, and review of the resulting plan by affected stakeholders. Natural extensions include richer service-area data, additional constraints, and larger instances, while retaining transparent reporting of assumptions.